# Marketing table for customer analysis

- Add a RFM score to the table
- Create the final table
- Insert data into the final table

In [0]:
WITH base AS (
  SELECT
    c.customer_id,
    CONCAT(c.firstname, ' ', c.lastname) AS fullname,
    c.create_date AS joining_date,
    MIN(s.order_date) AS first_order,
    MAX(s.order_date) AS last_order,
    COUNT(DISTINCT s.order_number) AS total_orders,
    SUM(s.sales) AS total_sales,
    SUM(s.quantity) AS total_quantity,
    ROUND(SUM(s.sales) / COUNT(DISTINCT s.order_number), 2) AS avg_order_value
  FROM data_lakehouse_project.gold.dim_customers AS c
  LEFT JOIN data_lakehouse_project.gold.fact_sales AS s
    ON c.customer_id = s.customer_id
  LEFT JOIN data_lakehouse_project.gold.dim_products AS p
    ON s.product_key = p.product_key
  GROUP BY ALL
),

top_category AS (
  SELECT customer_id, category, total_qty
  FROM (
    SELECT
      c.customer_id,
      p.category,
      SUM(s.quantity) AS total_qty,
      RANK() OVER(PARTITION BY c.customer_id ORDER BY SUM(s.quantity) DESC) AS rnk
    FROM data_lakehouse_project.gold.dim_customers AS c
    LEFT JOIN data_lakehouse_project.gold.fact_sales AS s 
      ON c.customer_id = s.customer_id
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p 
      ON s.product_key = p.product_key
    GROUP BY ALL
  )
  WHERE rnk = 1
),

top_subcategory AS (
  SELECT customer_id, sub_category, total_qty
  FROM (
    SELECT
      c.customer_id,
      p.sub_category,
      SUM(s.quantity) AS total_qty,
      RANK() OVER(PARTITION BY c.customer_id ORDER BY SUM(s.quantity) DESC) AS rnk
    FROM data_lakehouse_project.gold.dim_customers AS c
    LEFT JOIN data_lakehouse_project.gold.fact_sales AS s 
      ON c.customer_id = s.customer_id
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p 
      ON s.product_key = p.product_key
    GROUP BY ALL
  )
  WHERE rnk = 1
),

top_product AS (
  SELECT customer_id, product_name, total_qty
  FROM (
    SELECT
      c.customer_id,
      p.product_name,
      SUM(s.quantity) AS total_qty,
      RANK() OVER(PARTITION BY c.customer_id ORDER BY SUM(s.quantity) DESC) AS rnk
    FROM data_lakehouse_project.gold.dim_customers AS c
    LEFT JOIN data_lakehouse_project.gold.fact_sales AS s 
      ON c.customer_id = s.customer_id
    LEFT JOIN data_lakehouse_project.gold.dim_products AS p 
      ON s.product_key = p.product_key
    GROUP BY ALL
  )
  WHERE rnk = 1
)

SELECT
  b.*,
  tc.category AS top_category,
  tsc.sub_category AS top_subcategory,
  tp.product_name AS top_product
FROM base AS b
LEFT JOIN top_category AS tc 
  ON b.customer_id = tc.customer_id
LEFT JOIN top_subcategory AS tsc 
  ON b.customer_id = tsc.customer_id
LEFT JOIN top_product AS tp 
  ON b.customer_id = tp.customer_id